# Task 4.3 Effect of PARRM on ridge refit

Stimulation wrecks a ridge decoder trained on OFF trials, and refitting it on a few dozen
labelled ON trials recovers most of the accuracy. Two things could explain that recovery:
cleaning the stimulation artifact out of the input (PARRM), or simply having labels for
the session you are decoding. Task 3 tested cleaning *without* labels and it did nothing.
This notebook asks the harder question: once labels are on the table, does cleaning still
add anything?

The design is a paired comparison. For every draw from 4.1 — k calibration trials plus the
leftover ON query trials of one session — we refit a fresh ridge decoder twice: once on raw
features, once on PARRM-cleaned features. Same draws, same query trials, same penalty; the
only difference is what the input recording looks like.

**Headline: no.** k=2 → k=20 labelled trials is worth roughly +0.05 R² at 20 Hz; PARRM is
worth +0.0005 there and −0.001 at 3 Hz. The recovery lives in the labels, not in the
cleaning. Frequency-split numbers are at the bottom of the notebook. And an honest limit:
at R² this small (see the noise-floor caveat at the end) a 0.001 difference is not
measurable anyway, so the claim is "no detectable PARRM effect", not "PARRM is useless".

## What PARRM actually does

LFP: "Local Field Potential" — the summed electrical activity each electrode picks up from
the nearby neural population, filtered here to 0.1–300 Hz.

Butters performs a reaching task on a screen. Hand position is tracked by DeepLabCut from
an RGB side camera. Brain signals are captured by an implanted 64-electrode array
recording motor cortex LFP. During stimulation, a voltage is delivered through implanted
electrodes as a 3.5 kHz carrier, amplitude modulated (AM) at either 3 Hz or 20 Hz. The
recording hardware only partly demodulates the carrier, leaving a large residual at the
AM frequency on the LFP output as the **stimulation artifact** — far larger than the
neural signal around it.

The artifact's defining property is that it repeats: it is a fixed waveform restarting
every 1/f seconds, where f is the AM frequency — every 50 ms at 20 Hz (exactly one feature
bin) and every ~333 ms at 3 Hz (~6.7 bins). It is not random; it is clock-locked to the
stimulation.

PARRM (Dastin-van Rijn et al. 2021, Cell Reports Methods) exploits that repetition.
Average the recording at the same phase across many AM cycles: the neural and movement
related activity wanders from cycle to cycle and averages out, while the artifact sits in
the same place every cycle and survives — so the average is a template of the artifact
alone. Subtract the template from every sample and what remains is nominally artifact-free.
The key property for this task: PARRM needs no labels, no calibration trials, no
kinematics — you can clean a recording you know nothing about.

That invites a cheap fix: clean the ON recording, and a decoder trained on OFF days no
longer sees the artifact at all, so the ON gap closes by itself. Task 3.2 tested exactly
that; it did not happen. This notebook tests the weaker hope — once you refit on k labelled
ON trials anyway, does cleaning add anything on top?

Three reasons it shouldn't, which is what we see:

1. **The repetition is only approximate.** Artifact amplitude drifts within a session, so
   one template leaves residual — and the residual is still enormous next to the LFP
   (channel 64 in 250616-102853 reaches z ≈ 5300 under stimulation; hence the clipping
   below).
2. **The brain is clock-locked too.** Anything phase-locked to the stimulus — evoked
   potentials, movement entrained to the AM cycle — sits at the same phase as the artifact,
   so template subtraction removes it along with the artifact. PARRM cannot tell
   "artifact at phase θ" from "brain at phase θ".
3. **The refit already does this job.** With k labelled ON trials, ridge can learn small
   weights on the artifact-dominated channels by itself. Cleaning overlaps with what the
   labels were going to buy anyway.

## Why ridge, and what k is counting

Ridge is the cheapest adaptive decoder there is: a linear map from 350 features (10 lagged
bins × 35 channels) to the two velocity components, fitted in closed form. That is what
makes it the right probe here — if a plain linear refit on k ON trials already delivers
what PARRM promises, the effect lives in the data, not the model, and no deep decoder is
needed to see it.

What each draw hands to the refit:

- A trial contributes ~34 valid bins, so k=2 is ~68 rows against 350 features and k=20 is
  ~680. k is a dose–response axis: same session, same query definition, only the number of
  labelled trials changes. Expect k=2 to sit at R² ≈ 0 — hopelessly underdetermined, ridge
  falls back to the mean. Measured: 0.000 at 3 Hz, 0.008 at 20 Hz.
- λ is chosen once on the ON validation sessions and frozen for every draw, so the k=2
  point fails for lack of labelled data, not for a failed hyperparameter search.
- Each fit produces two scores. `r2` is on the draw's leftover ON query trials; `r2_off`
  puts the same decoder on that day's no-stimulation recording, which it never saw.
  `r2_off` is the control that says the refit learned kinematics rather than this
  session's artifact skin: it rises in step with `r2` (0.003 → 0.038 at 20 Hz), staying
  within 0.005 (3 Hz) to 0.024 (20 Hz) of the ON score.

## Setup


In [ ]:
import os
import sys

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
TASK3_DIR = os.path.join(os.path.dirname(TASK4_DIR), 'task3')
for path in (TASK4_DIR, TASK3_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

import utils
import task3_common as C  # ridge solver, penalty grid, clipping and PARRM map shared with task 3

print('task 4 helpers:', utils.TASK4_DIR)
print('task 3 helpers:', C.TASK_DIR)
print('draws file    :', utils.DRAWS_PATH)


In [ ]:
import numpy as np
import pandas as pd

LAG_COUNT = utils.LAG_COUNT
FEATURE_VARIANTS = ((utils.RAW_VARIANT, 'raw'), (utils.PARRM_VARIANT, 'parrm'))
DRAWS = utils.load_json(utils.DRAWS_PATH)

def clip(windows):
    X, y = windows
    return np.clip(X, -C.CLIP_Z, C.CLIP_Z), y

print('draws:', len(DRAWS['sessions']), 'sessions | ks', DRAWS['ks'],
      '| query definition', DRAWS.get('query_definition'))


## Protocol

Everything comes from `task3_common` — the ridge solver, the penalty grid (`C.LAMBDAS`),
the clipping (`C.CLIP_Z`) — so task 3 and task 4.3 make identical choices and only the fit
data differs. Features are clipped to ±10 so one artifact bin cannot dominate the weights.
The clip binds almost only on raw features under stimulation, so if anything it handicaps
the raw arm — and PARRM still does not beat it.

Beyond that: λ is picked once on the ON validation sessions and held for every draw; each
draw normalises on its own trials (see 4.1); raw and PARRM share the same draws and query
trials. Each of those choices removes a confound, leaving k and the input variant as the
only things that vary — which is the experiment.

In [ ]:
# Same protocol as task 3: fit on the OFF training windows, score on the ON validation
# query trials, and keep the lambda with the best mean validation R2.
X_train, y_train = clip(utils.concat_session_windows(utils.OFF_TRAINING_SESSIONS, 'train', utils.RAW_VARIANT, LAG_COUNT))
validation = [clip(utils.windows_from_valid_bins(utils.load_split_dataset(s, 'query', utils.RAW_VARIANT, LAG_COUNT)))
              for s in utils.sessions_in_split('val', utils.STIMULATED_ON)]

fitter = C.RidgeFitter(X_train, y_train)
scores = {float(lam): np.mean([utils.mean_r2(y, fitter.fit(lam).predict(X)) for X, y in validation])
          for lam in C.LAMBDAS}
alpha = max(scores, key=scores.get)
print('ridge alpha:', alpha, '(from task 3 grid of', len(C.LAMBDAS), ')')
pd.Series(scores).round(4)


In [ ]:
def refit_draw(session, k, index, draw, variant, feature_label, alpha):
    X_cal, y_cal = clip(utils.drawn_trial_windows(session, draw, variant, LAG_COUNT))
    X_query, y_query = clip(utils.query_trial_windows(session, draw, variant, LAG_COUNT))
    off = clip(utils.windows_from_valid_bins(utils.load_same_day_off_dataset(session, variant, LAG_COUNT)))
    model = C.ridge_path(X_cal, y_cal, [alpha])[0]  # task 3 solver, one lambda
    return utils.score_window_decoder(model, session, 'ridge', feature_label,
                           (X_query, y_query), off, k=k, draw=index)

rows = []
for session in DRAWS['sessions']:
    for k, index, draw in utils.iter_draws(DRAWS, session):
        for variant, feature_label in FEATURE_VARIANTS:
            rows.append(refit_draw(session, k, index, draw, variant, feature_label, alpha))
    print('scored', session)

results = pd.DataFrame(rows)
results.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_3_parrm_ridge.csv'), index=False)
print(results.shape)
results.groupby(['feature_variant', 'k'])[['r2_ave', 'r2_off_ave']].mean().round(3)


## Recovery per k

Each row below is one (variant, k): mean over 14 sessions × 10 draws, with a 95 % CI from
resampling sessions — draws inside a session are nested and correlated, so resampling them
would understate the spread. `r2_off` is the OFF control from above.

Read the k axis, not the gap between rows: a PARRM-vs-raw difference of 0.001 is invisible
at this scale, while the k=2 → k=20 rise (~0.03 at 3 Hz, ~0.05 at 20 Hz) is 30–50× larger.

In [ ]:
per_session = (results.groupby(['feature_variant', 'k', 'session'], as_index=False)
               .agg(r2=('r2_ave', 'mean'), r2_off=('r2_off_ave', 'mean')))

summary = []
for (variant, k), part in per_session.groupby(['feature_variant', 'k']):
    low, high = utils.bootstrap_confidence_interval(part.r2)
    summary.append(dict(feature_variant=variant, k=k, sessions=len(part),
                        r2=part.r2.mean(), ci_low=low, ci_high=high,
                        r2_off=part.r2_off.mean()))
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_3_summary.csv'), index=False)
summary.round(3)

## What the numbers say, split by frequency

Pooled above, the two variants look identical — that is the honest headline. But pooling
mixes two rather different k-curves, so split them.

**3 Hz — 7 sessions (days 250520, 250609, 250613)**

| k | raw R² | PARRM R² | PARRM − raw | SE (sessions) | t |
|--:|------:|--------:|------------:|---:|---:|
| 2 | 0.0003 | 0.0001 | −0.0002 | 0.0001 | −1.8 |
| 5 | 0.0103 | 0.0099 | −0.0004 | 0.0003 | −1.4 |
| 10 | 0.0185 | 0.0179 | −0.0006 | 0.0005 | −1.3 |
| 20 | 0.0292 | 0.0280 | −0.0012 | 0.0007 | −1.7 |

**20 Hz — 7 sessions (days 250530, 250616)**

| k | raw R² | PARRM R² | PARRM − raw | SE (sessions) | t |
|--:|------:|--------:|------------:|---:|---:|
| 2 | 0.0082 | 0.0083 | +0.0001 | 0.0002 | +0.3 |
| 5 | 0.0256 | 0.0258 | +0.0002 | 0.0002 | +0.9 |
| 10 | 0.0426 | 0.0430 | +0.0004 | 0.0004 | +1.0 |
| 20 | 0.0619 | 0.0624 | +0.0005 | 0.0004 | +1.1 |

The SE and t take the **session** as the unit (n=7), not the draw. The 10 draws per session
are nested; treating them as independent would shrink the error bars by √10 and
manufacture significance.

1. **PARRM does nothing, in either direction.** Every |t| ≤ 1.8, and the sign even flips
   between frequencies (−0.0012 at 3 Hz, +0.0005 at 20 Hz) — which is what "no effect"
   looks like, not a real frequency-dependent one. Against the k effect it is a rounding
   error: the refit gains 0.054 R² at 20 Hz from k=2 to k=20; PARRM adds 0.0005.
2. **k behaves like a clean dose–response.** Monotone at both frequencies, for both
   variants, at every step. k=2 buys nothing; k=20 lifts the ON score to roughly where the
   same decoder sits on clean data (r2_off rises 0.003 → 0.038 at 20 Hz) — the refit is
   recovering kinematics, not artifact quirks. At 20 Hz / k=20 the ON score actually
   finishes 0.024 *above* the OFF score, and that cross-recording gap is ~50× the PARRM
   effect being measured here; another reason 0.001-scale claims are not trustworthy.
3. **The 3 Hz vs 20 Hz gap is not a PARRM story, and may not be a frequency story.**
   20 Hz recovers to 0.062 and 3 Hz to 0.029, about double — but frequency is confounded
   with day here: all seven 3 Hz sessions sit on 250520/250609/250613 and all seven 20 Hz
   sessions on 250530/250616, and 250616 is also the most decodable day overall (within-day
   OFF ceiling 0.126 vs 0.039–0.083, `results/diagnostic_within_day_off.csv`). Two days
   against three cannot separate frequency from day, so read the gap as "those sessions
   were easier", not "20 Hz is easier to recover from".

If the gap is real, one mechanism fits and is worth keeping for later: at 20 Hz the
artifact period equals one 50 ms bin, so every bin samples the artifact at the same phase
and it acts like a constant offset — exactly what the ridge intercept, estimated from the k
calibration trials, absorbs for free. At 3 Hz the period is ~6.7 bins, so the artifact
phase sweeps across bins within a trial and looks like fast structure a linear map cannot
fold into an intercept. The prediction — the 20 Hz curve starts and finishes higher — holds
(k=2: 0.008 vs 0.0003). Testing it properly needs a spread of AM frequencies on the same
days; the dataset does not contain that spread, which is the actual blocker.

### Caveats to keep attached to these numbers

- **R² here is at the noise floor.** Fitting and testing on different trials of the *same*
  no-stim session gives R² ≈ 0.067 (0.039–0.126 across days), so the best number in these
  tables (0.062) already sits at that ceiling and the per-bin velocity target — not the
  artifact — is the limiting factor. This dataset cannot resolve a 0.001 R² difference.
- **Clipping handicaps the raw arm.** It binds almost only on raw features under
  stimulation, so the raw arm loses information the cleaned arm keeps — and PARRM still
  fails to beat it. That strengthens the negative result.
- **Draws are nested** across k within a session, so the four points of a curve are not
  independent: trust the shape of the curve, not the spacing between adjacent points.
- **λ is shared across variants.** Per-draw re-standardisation mostly neutralises the
  scale difference, but if PARRM shifts the effective feature scale, the optimal penalty
  shifts too. A per-variant λ check is cheap insurance before generalising.
- **4.3's role in the task 4 story is as a control, and a negative one.** 4.1–4.2
  establish that labelled ON trials close the gap; 4.3 shows the cleaning term contributes
  ~0 even once labels are present, so the credit belongs to the labels.